# Image embeddings with VectorMesh

This notebook shows how to turn **images** into cached embedding vectors with `ImageVectorizer`, and how those vectors plug into the *exact same* downstream pipeline as the text / regex vectorizers (`Collate` + `Serial`).

The idea is **embed once, train many times**: a (possibly heavy) vision model runs *once* to fill a `VectorCache`; afterwards you only ever train a small head on the cached vectors, which is cheap on CPU. You can even precompute the cache on a GPU and distribute it to people without one.

You pick the image dataset from a small **catalog** below by setting one variable, `choice`. Because the column names are auto-detected with `DatasetSchema`, switching datasets needs no other change — nothing downstream cares what the columns are called. The catalog ranges from a near-trivial binary task (dog vs food) up to a 102-class flowers problem, so you can watch the *same* pipeline behave very differently.

In [ ]:
from pathlib import Path

from datasets import load_dataset

from vectormesh import ImageVectorizer, VectorCache
from vectormesh.data import DatasetSchema, OneHot

## Choosing an embedding model

`ImageVectorizer` is model-agnostic: it wraps any HuggingFace vision model that loads with `AutoModel` + `AutoImageProcessor`. Bigger models give richer embeddings but cost more to run (once!). A spread from tiny CNNs to distilled vision transformers:

| model | params | type | emb dim | notes |
|---|---|---|---|---|
| `google/mobilenet_v2_1.0_224` | 3.5M | CNN | 1280 | tiniest, **default here** |
| `google/efficientnet-b0` | ~5.3M | CNN | 1280 | efficient CNN |
| `microsoft/resnet-18` | 11.7M | CNN | 512 | classic, on-topic for convolutions |
| `facebook/dinov2-small` | 22.1M | ViT (distilled) | 384 | strong embeddings for its size |
| `microsoft/resnet-50` | 25.6M | CNN | 2048 | |
| `facebook/dinov2-base` | 86.6M | ViT (distilled) | 768 | precompute on GPU, distribute cache |
| `facebook/dinov2-large` | 304M | ViT | 1024 | heavy |

The `dinov2-small/base/large` models are themselves *distillations* of the 1.1B DINOv2-giant — small but smart. None of these need a GPU at *inference* time; model size only changes how long the one-time caching takes.

In [ ]:
models = {
    "mobilenet_v2": "google/mobilenet_v2_1.0_224",  # 3.5M   CNN, dim 1280
    "efficientnet_b0": "google/efficientnet-b0",     # ~5.3M  CNN, dim 1280
    "resnet18": "microsoft/resnet-18",               # 11.7M  CNN, dim 512
    "dinov2_small": "facebook/dinov2-small",         # 22.1M  ViT, dim 384
    "resnet50": "microsoft/resnet-50",               # 25.6M  CNN, dim 2048
    "dinov2_base": "facebook/dinov2-base",           # 86.6M  ViT, dim 768
}

# Default: the smallest model, so this notebook runs fast on a laptop CPU.
# Swap in a bigger one (e.g. models["dinov2_small"]) and re-run — nothing else changes.
model_name = models["mobilenet_v2"]
model_name

## The data

Pick a dataset by setting `choice` to a key of `datasets_catalog`. Only the hub id (and, rarely, a list of extra columns to drop) differs between entries — the **column names are auto-detected** by `DatasetSchema.infer`, so you never need to know whether the label is called `label`, `labels` or `target`, or the image `image` vs `img`. For a dataset with unusual names, override with `DatasetSchema.infer(dataset["train"], input_col=..., label_col=...)`.

We subsample to keep the one-time embedding pass quick; drop the `.select(...)` calls to use the full splits. Not every dataset ships a dedicated `test` split, so we fall back to carving an eval split off `train`.

In [ ]:
# A small catalog of image-classification datasets. To switch, change `choice`
# below — nothing else needs to change, because DatasetSchema detects the columns.
#   drop: non-image columns to also strip from the cache (beyond the image itself)
datasets_catalog = {
    "flowers":             {"hub": "nkirschi/oxford-flowers",    "drop": []},        # 102 classes, harder
    "dog_food":            {"hub": "sasha/dog-food",             "drop": []},        # 2 classes, near-trivial, fastest
    "rock_paper_scissors": {"hub": "Javtor/rock-paper-scissors", "drop": []},        # 3 classes, photos of hands
    "eurosat":             {"hub": "blanchon/EuroSAT_RGB",       "drop": ["filename"]},  # 10 classes, 27k satellite tiles
}

choice = "flowers"  # <- pick any key from the catalog above
# str(...): each entry's "hub" is always a string and "drop" always a list, but a plain
# dict literal doesn't let the type checker narrow per-key, so it infers `str | list[str]`
# for every value regardless of which key produced it.
dataset_name = str(datasets_catalog[choice]["hub"])
extra_drop = datasets_catalog[choice]["drop"]
tag = dataset_name.split("/")[-1]

dataset = load_dataset(dataset_name, cache_dir=f"../assets/{tag}/")
dataset

In [ ]:
schema = DatasetSchema.infer(dataset["train"])
# override for an odd dataset, e.g. DatasetSchema.infer(dataset["train"], label_col="fine_label")

schema

In [ ]:
labels = dataset["train"].features[schema.label_col].names

# Not every dataset ships a dedicated eval split; fall back to carving one off train.
eval_split = next((s for s in ("test", "validation", "valid") if s in dataset), None)
if eval_split is None:
    parts = dataset["train"].train_test_split(test_size=0.2, seed=42)
    trainset, evalset = parts["train"], parts["test"]
else:
    trainset, evalset = dataset["train"], dataset[eval_split]

# subsample for a quick demo — remove the .select(...) to use the full splits
train = trainset.select(range(min(2000, len(trainset))))
test = evalset.select(range(min(1000, len(evalset))))
print(schema)
print(f"{len(labels)} labels | train: {len(train)} | test: {len(test)}")

Each row has an `image` (a PIL image) and an integer label:

In [ ]:
item = train[0]
image = item[schema.input_col]
print(f"label: {labels[item[schema.label_col]]}, size: {image.size}, mode: {image.mode}")
image

We one-hot encode the label so it matches the `Collate` target used downstream (same as in the text notebooks).

In [ ]:
# label_col is Optional on DatasetSchema (some datasets genuinely have none to detect);
# every catalog entry here is a labelled classification dataset, so it must be set.
assert schema.label_col is not None, f"{tag} has no detectable label column"
onehot = OneHot(num_classes=len(labels), label_col=schema.label_col, target_col="onehot")
train = train.map(onehot)
test = test.map(onehot)

## Embedding the images

`ImageVectorizer` reads from `schema.input_col` (the detected image column) and writes one embedding vector per image into the `embed` column. The embedding dimension is discovered automatically from the model with a dummy forward pass.

In [ ]:
vectorizer = ImageVectorizer(model_name=model_name, input_col=schema.input_col)
vectorizer.get_hidden_size

### About that "LOAD REPORT"

Loading the model prints a report with a couple of `UNEXPECTED` keys, e.g.:

```
[transformers] MobileNetV2Model LOAD REPORT from: google/mobilenet_v2_1.0_224
Key               | Status     |
------------------+------------+
classifier.bias   | UNEXPECTED |
classifier.weight | UNEXPECTED |
```

This is **not an error** — it is a confirmation of exactly what we want. The checkpoint is a *classification* model: a **backbone** (the conv feature extractor) plus a **`classifier`** head mapping features → 1000 ImageNet classes. `ImageVectorizer` loads it with `AutoModel`, which builds the **base model without any task head**. So the `classifier` weights in the file have nowhere to go and are reported as `UNEXPECTED` — i.e. *"these weights were in the checkpoint but the model didn't use them."*

That is the whole point of using the model as an embedder: we keep the backbone's `pooler_output` *vector* and drop the ImageNet *head*.

- `UNEXPECTED` (here) → benign: extra weights in the file we don't need.
- `MISSING` (the opposite) → *would* be the worrying one: backbone weights not found in the checkpoint, leaving parts of the model randomly initialised.

If you actually wanted the pretrained classification head, you'd load `AutoModelForImageClassification` instead. For embeddings, this report is just reassurance that the head was correctly discarded. (To silence it, call `from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()` before building the vectorizer.)

## Caching

`VectorCache.create` runs the model over the dataset **once** and stores the result on disk. We pass `remove_columns=[schema.input_col, *extra_drop]` to drop the raw pixels (and any dataset-specific extras, like EuroSAT's `filename`) from the cache — the resulting folder only holds the (small) vectors + labels, which makes it cheap to share. Drop that argument if you want to keep the images around.

This is the slow step (it runs the encoder over every image); everything after it is fast. In a real project you would `VectorCache.load(...)` an existing cache instead of recreating it.

In [ ]:
traincache = VectorCache.create(
    cache_dir=Path("tmp/artefacts"),
    vectorizer=vectorizer,
    dataset=train,
    dataset_tag=f"{tag}_train",
    remove_columns=[schema.input_col, *extra_drop],
)
testcache = VectorCache.create(
    cache_dir=Path("tmp/artefacts"),
    vectorizer=vectorizer,
    dataset=test,
    dataset_tag=f"{tag}_test",
    remove_columns=[schema.input_col, *extra_drop],
)
traincache.metadata

Each observation is now a `(dim,)` embedding plus the one-hot target — the raw image is gone:

In [ ]:
row = traincache[0]
print("columns:", traincache.column_names)
print("embed:", tuple(row["embed"].shape), "| onehot:", tuple(row["onehot"].shape))

## Augmenting in feature space

Pixel-space augmentation (random crops/flips) wants a *fresh* random view every epoch — which fights the cache, because each new view would mean re-running the frozen encoder. Instead we augment the **cached embeddings** directly with `GaussianNoise`: cheap, fresh on every step, and a good regulariser for the small head — especially when training on few images.

`GaussianNoise` is active only in `train()` mode and is a no-op during `eval()`, so validation stays deterministic. It acts on the last dimension, so it drops straight into a `Serial` pipeline in front of the classifier head.

In [ ]:
import torch
from torch.utils.data import DataLoader

from vectormesh.components import GaussianNoise, NeuralNet, Serial
from vectormesh.data import Collate

hidden_size = traincache.metadata["embed"]["hidden_size"]
collate_fn = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
trainloader = DataLoader(traincache, batch_size=32, shuffle=True, collate_fn=collate_fn)
testloader = DataLoader(testcache, batch_size=32, shuffle=False, collate_fn=collate_fn)

pipeline = Serial([
    GaussianNoise(sigma=0.1),  # feature-space augmentation (train-only)
    NeuralNet(hidden_size=hidden_size, out_size=len(labels)),
])

X, y = next(iter(trainloader))
print("batch X:", tuple(X.shape), "| y:", tuple(y.shape), "| logits:", tuple(pipeline(X).shape))

Quick sanity check that the noise is on during training and off during eval:

In [ ]:
pipeline.train()
noisy_a, noisy_b = pipeline(X), pipeline(X)
pipeline.eval()
clean_a, clean_b = pipeline(X), pipeline(X)
print("train-mode varies between runs:", not torch.allclose(noisy_a, noisy_b))
print("eval-mode is deterministic:    ", torch.allclose(clean_a, clean_b))

## Training the head

Now we train just the small head on the frozen embeddings — fast, even on CPU. We reuse `mltrainer`, with `CrossEntropyLoss` and two metrics, `Accuracy` and `F1Score`. Both accept the one-hot targets produced by `Collate`: `Accuracy` reads them as soft targets and compares `argmax`es, while `F1Score` works element-wise.

In [ ]:
import torch.optim as optim
from mltrainer import ReportTypes, Trainer, TrainerSettings

from vectormesh.components.metrics import Accuracy, F1Score

settings = TrainerSettings(
    epochs=2, # increase this number to train longer and see better results
    metrics=[Accuracy(), F1Score()],
    logdir=Path("logs"),
    train_steps=len(trainloader),
    valid_steps=len(testloader),
    reporttypes=[ReportTypes.TOML],
)

trainer = Trainer(
    model=pipeline,
    settings=settings,
    loss_fn=torch.nn.CrossEntropyLoss(),
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=testloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device="cpu",  # the head is tiny and the vectors are cached
)
trainer.loop()

## Exercises

**1. Swap the embedding model.** Change `model_name` (e.g. `models["dinov2_small"]`) and re-run. Does a stronger encoder give a better accuracy on the same head? How much slower is the caching step? The effect is easiest to *see* on a hard dataset — `choice = "flowers"` (102 classes) leaves plenty of headroom, whereas `dog_food` is already near-perfect with the tiny default model.

**2. Tune the augmentation.** Try `GaussianNoise(sigma=0.0)` (off), `0.05`, `0.2`, `0.5`. Where does noise help, and where does it start hurting? Try shrinking the training set (e.g. `train.select(range(200))`) — does noise matter more with fewer images?

**3. Try other datasets.** Just change `choice` at the top — `DatasetSchema` handles the columns and the split fallback handles missing `test` splits. What to expect per catalog entry:

| `choice` | classes | what you'll observe |
|---|---|---|
| `dog_food` | 2 (dog / food) | near-trivial — accuracy/F1 shoot to ~1.0 within an epoch, even on the smallest model |
| `rock_paper_scissors` | 3 (hands) | easy but not trivial; a good sanity check that the pipeline generalises |
| `flowers` | 102 | genuinely hard — the place to feel the difference between encoders in exercise 1 |
| `eurosat` | 10 (satellite) | 27k tiles, so keep the `.select(...)` subsample or precompute on GPU; its extra `filename` column is already handled via the catalog's `drop` list |

For a dataset with unusually named columns, pin them: `DatasetSchema.infer(dataset["train"], input_col=..., label_col=...)`.

## Cleanup

Remove the demo cache (we only made it to keep the notebook self-contained):

In [ ]:
import shutil

shutil.rmtree("tmp/artefacts", ignore_errors=True)